# รายงานผลการดำเนินงาน Sprint 1 — Final Term Project

## ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)

**รายวิชา:** CP352301 Script Programming (1/2569) · **Sprint 1:** Front-End App Dev (สัปดาห์ที่ 12)
**นำเสนอ:** 15–16/9/69 · **ส่งงาน:** 18/9/69

| บทบาท (หมุนเวียนตาม Sprint) | สมาชิกในทีม |
|---|---|
| Planner / Team Leader | [ชื่อนักศึกษา] |
| Coder | [ชื่อนักศึกษา] |
| Debugger / QA | [ชื่อนักศึกษา] |

**Repository / Pull Request:** _(จะเพิ่มลิงก์เมื่อสร้าง Pull Request บน GitHub)_

---

โน้ตบุ๊กนี้เป็นรายงาน Sprint 1 ตามวงรอบ **Plan → Execution → Review** รันได้ครบวงจรทันที
โดยไม่ต้องใช้คีย์ API หรืออินเทอร์เน็ต (สาธิตด้วยข้อมูลตัวอย่างในตัวโปรแกรม)

**โครงสร้างรายงาน:** ส่วนที่ 1 แผนงาน · ส่วนที่ 2 การพัฒนา · ส่วนที่ 3 ผลลัพธ์และการทดสอบ


## ส่วนที่ 1 — แผนงาน (Plan)

### 1.1 เป้าหมายและขอบเขตของ Sprint 1

**เป้าหมาย:** พัฒนาส่วนปฏิสัมพันธ์กับผู้ใช้ (Presentation Layer) ของโปรแกรม — ข้อความต้อนรับ ·
เมนูคำสั่ง · การรับและแปลงอินพุต (`.strip().lower()`) · การตรวจสอบความถูกต้องของข้อมูลนำเข้า
โดยโปรแกรมต้องไม่หยุดทำงานเมื่อผู้ใช้ป้อนข้อมูลผิดพลาด

**ขอบเขต Sprint 1:** CLI แบบพิมพ์คำสั่ง · เมนู · การตรวจสอบอินพุต · การจัดการข้อผิดพลาด · ชุดทดสอบอัตโนมัติ

**นอกขอบเขต (Sprint 2 เป็นต้นไป):** TMDB API จริง · ฐานข้อมูล SQLite · อัลกอริทึมค้นหา/กรอง/เรียงลำดับ · ส่งออก CSV
(คำสั่ง search / discover สาธิตด้วยข้อมูลตัวอย่างในตัวโปรแกรม)

### 1.2 คำสั่งของระบบ

| คำสั่ง | อินพุต | พฤติกรรมที่คาดหวัง |
|---|---|---|
| `search <ชื่อเรื่อง>` | ข้อความ | ค้นหาจากข้อมูลตัวอย่าง · ไม่พบ → ข้อความแจ้ง |
| `discover <รหัส>` | จำนวนเต็ม > 0 | แสดงภาพยนตร์ที่คล้ายกัน เรียงตามคะแนน |
| `watchlist add <รหัส>` | จำนวนเต็ม > 0 | เพิ่มเข้ารายการรับชม (ป้องกันเพิ่มซ้ำ) |
| `watchlist list` / `clear` | — | แสดง / ล้างรายการรับชม |
| `help` | — | แสดงคำสั่งทั้งหมด |
| `quit` / `exit` / `q` / `ออก` | — | ออกทันที (ไม่สนใจตัวพิมพ์เล็ก–ใหญ่/ช่องว่าง) |

### 1.3 Definition of Done (ผลตรวจครบในส่วนที่ 3)

- [ ] พิมพ์ `quit` / `QUIT` / `Quit` / `  quit  ` → ออกทันทีพร้อมข้อความอำลา
- [ ] คำสั่งไม่รู้จัก → แจ้งเตือนและกลับสู่เมนู (โปรแกรมไม่หยุดทำงาน)
- [ ] บรรทัดว่างหรือช่องว่างล้วน → แจ้งเตือน "กรุณาพิมพ์คำสั่ง"
- [ ] รหัสภาพยนตร์ที่ไม่ใช่ตัวเลข (`abc`) → แจ้งเตือน ไม่ crash (try / except ValueError)
- [ ] รหัสภาพยนตร์ติดลบหรือศูนย์ → ปฏิเสธพร้อมข้อความ
- [ ] อินพุตสิ้นสุด (EOF) หรือ Ctrl+C → ออกจากโปรแกรมอย่างสุภาพ
- [ ] โค้ดผ่าน flake8 และมี docstring ครบทุกเมธอด


## ส่วนที่ 2 — การพัฒนา (Execution)

เซลล์ด้านล่างสร้างโครงสร้างโปรเจกต์ตาม `PLAN.md` เขียนโค้ดที่ส่งมอบ แล้วสาธิตการทำงานจริง
ด้วยการจำลองอินพุต — ผลลัพธ์ทั้งหมดเป็นผลรันจริงจากโค้ดด้านบน


In [1]:
# เตรียมโครงสร้างโปรเจกต์และย้ายเข้าไปทำงานในโฟลเดอร์
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here if here.name == "script-final-project" else here / "script-final-project"
for folder in ("src", "tests", "data", "notebooks"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working directory:", ROOT)
print("python:", sys.version.split()[0])


working directory: C:\Users\User\AppData\Local\Temp\sfp_nbcheck\workdir\script-final-project
python: 3.11.15


In [2]:
# ติดตั้งไลบรารีที่ใช้ทดสอบ (มีอยู่แล้วในเครื่องจะข้ามทันที)
%pip install -q pytest flake8
import pytest
print("pytest", pytest.__version__)


Note: you may need to restart the kernel to use updated packages.
pytest 9.1.1


In [3]:
%%writefile requirements.txt
requests>=2.31
pytest>=8.0
flake8>=7.0


Writing requirements.txt


In [4]:
%%writefile setup.cfg
[flake8]
max-line-length = 99
exclude = .venv,__pycache__,.git,.pytest_cache

[tool:pytest]
testpaths = tests


Writing setup.cfg


In [5]:
%%writefile PLAN.md
# PLAN.md — แผนงาน Sprint 1 (Front-End App Dev)

**โปรเจกต์:** ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)
**รายวิชา:** CP352301 Script Programming (1/2569) — Final Project
**Sprint 1:** สัปดาห์ที่ 12 — นำเสนอ 15–16/9/69 · ส่งงาน 18/9/69
**สถานะ:** สเปกต์ฉบับสมบูรณ์ (นิยาม Definition of Done เรียบร้อยก่อนเริ่มเขียนโค้ด)

---

## 1. เป้าหมายของ Sprint 1

ออกแบบและพัฒนาส่วนปฏิสัมพันธ์กับผู้ใช้ (Presentation Layer) ของโปรแกรม:

- ข้อความต้อนรับ (welcome banner) และเมนูคำสั่ง (menu)
- การรับคำสั่งและแปลงอินพุต — ตัดช่องว่างและแปลงตัวพิมพ์เล็กด้วย `.strip().lower()`
- การตรวจสอบความถูกต้องของข้อมูลนำเข้า (Input Validation) ครอบคลุมกรณีขอบเขต
- ควบคุมการทำงานหลักด้วย `while` loop และจัดการข้อผิดพลาดด้วย `try / except`

## 2. ขอบเขต (Scope)

| ประเภท | รายการ |
|---|---|
| ในขอบเขต | CLI แบบพิมพ์คำสั่ง (welcome · menu · search · discover · watchlist · help · quit) · การตรวจสอบอินพุต · การจัดการข้อผิดพลาด · ชุดทดสอบอัตโนมัติของ CLI |
| นอกขอบเขต (Sprint 2 เป็นต้นไป) | การเชื่อมต่อ TMDB API จริง · ฐานข้อมูล SQLite · อัลกอริทึมค้นหา/กรอง/เรียงลำดับจริง · การส่งออก CSV |
| หมายเหตุ | คำสั่ง search / discover สาธิตด้วย "ข้อมูลตัวอย่าง" ในตัวโปรแกรม เพื่อทดสอบส่วนติดต่อผู้ใช้ก่อนเชื่อมระบบจริงใน Sprint 2 |

## 3. คำสั่งของระบบ (Command Specification)

| คำสั่ง | อินพุต | พฤติกรรมที่คาดหวัง |
|---|---|---|
| `search <ชื่อเรื่อง>` | ข้อความ | ค้นหาจากข้อมูลตัวอย่าง แสดงเป็นรายการ · ไม่พบ → ข้อความแจ้ง |
| `discover <รหัส>` | จำนวนเต็ม > 0 | แสดงภาพยนตร์ที่คล้ายกัน (เรียงตามคะแนน) จากข้อมูลตัวอย่าง |
| `watchlist add <รหัส>` | จำนวนเต็ม > 0 | เพิ่มเข้ารายการรับชม (ป้องกันเพิ่มซ้ำ) |
| `watchlist list` | — | แสดงรายการรับชมปัจจุบัน |
| `watchlist clear` | — | ล้างรายการรับชม |
| `help` | — | แสดงคำสั่งทั้งหมด |
| `quit` / `exit` / `q` / `ออก` | — | ออกจากโปรแกรมทันที (ไม่สนใจตัวพิมพ์เล็ก–ใหญ่/ช่องว่าง) |

## 4. Definition of Done (DoD)

- [ ] พิมพ์ `quit`, `QUIT`, `Quit`, `  quit  ` → ออกจากโปรแกรมทันที พร้อมข้อความอำลา
- [ ] คำสั่งที่ไม่รู้จัก → แจ้งเตือนและกลับสู่เมนู (โปรแกรมไม่หยุดทำงาน)
- [ ] บรรทัดว่างหรือช่องว่างล้วน → แจ้งเตือน "กรุณาพิมพ์คำสั่ง"
- [ ] รหัสภาพยนตร์ที่ไม่ใช่ตัวเลข (`abc`) → แจ้งเตือน ไม่ crash (try / except ValueError)
- [ ] รหัสภาพยนตร์ติดลบหรือศูนย์ (`-3`, `0`) → ปฏิเสธพร้อมข้อความ
- [ ] อินพุตสิ้นสุด (EOF) หรือกด Ctrl+C → ออกจากโปรแกรมอย่างสุภาพ
- [ ] โค้ดผ่าน flake8 และมี docstring ครบทุกเมธอด

## 5. บทบาททีม (หมุนเวียนตามรอบ Sprint)

| บทบาท | ผู้รับผิดชอบ |
|---|---|
| Planner / Team Leader | [ชื่อนักศึกษา] |
| Coder | [ชื่อนักศึกษา] |
| Debugger / QA | [ชื่อนักศึกษา] |

## 6. แผนการทดสอบ (Test Plan)

ทดสอบกรณีขอบเขตตาม DoD สองช่องทาง:
1. ชุดทดสอบอัตโนมัติด้วย `pytest` (จำลองอินพุต ไม่ต้องใช้ terminal จริง)
2. การจำลองเซสชันจริงด้วยอินพุตสคริปต์ — ผลลัพธ์สรุปอยู่ในรายงาน Sprint 1 (notebook)


Writing PLAN.md


In [6]:
%%writefile src/__init__.py
"""Movie Discovery & Watchlist Builder — core package."""

__version__ = "0.1.0"


Writing src/__init__.py


In [7]:
%%writefile src/sample_data.py
"""Built-in sample dataset used by the Sprint 1 CLI and the test-suite.

The TMDB-backed data layer arrives in Sprint 2; until then the CLI is
demonstrated safely offline with these five movies.
"""

SAMPLE_MOVIES = [
    {"id": 101, "title": "Alpha Signal", "release_date": "2010-05-01",
     "vote_average": 7.8, "vote_count": 1200, "popularity": 40.0},
    {"id": 102, "title": "Bravo Horizon", "release_date": "2014-09-12",
     "vote_average": 6.1, "vote_count": 300, "popularity": 22.0},
    {"id": 103, "title": "Cobalt Night", "release_date": "1999-01-20",
     "vote_average": 8.4, "vote_count": 2100, "popularity": 55.0},
    {"id": 104, "title": "Delta Echo", "release_date": "",
     "vote_average": 5.0, "vote_count": 50, "popularity": 5.0},
    {"id": 105, "title": "Emerald Run", "release_date": "2021-11-05",
     "vote_average": 8.0, "vote_count": 900, "popularity": 30.0},
]


Writing src/sample_data.py


In [8]:
%%writefile src/cli.py
"""Interactive command-line front-end for the Movie Discovery app.

Sprint 1 scope (presentation layer only): welcome banner, command menu,
input handling (``.strip().lower()``), input validation, and graceful
exits.  ``search`` / ``discover`` are simulated with the built-in sample
dataset; the TMDB-backed services arrive in Sprint 2.
"""

from .sample_data import SAMPLE_MOVIES

WELCOME_MESSAGE = """\
============================================================
  Movie Discovery & Watchlist Builder — CLI (Sprint 1)
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
============================================================"""

MENU_TEXT = """\
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>      ค้นหาภาพยนตร์จากชื่อเรื่อง (ข้อมูลตัวอย่าง)
  discover <รหัส>         แสดงภาพยนตร์ที่คล้ายกัน (ข้อมูลตัวอย่าง)
  watchlist add <รหัส>    เพิ่มภาพยนตร์เข้ารายการรับชม
  watchlist list          แสดงรายการรับชม
  watchlist clear         ล้างรายการรับชม
  help                    แสดงคำสั่งทั้งหมด
  quit / exit / ออก       ออกจากโปรแกรม"""

FAREWELL_MESSAGE = "ขอบคุณที่ใช้งานโปรแกรม ลาก่อน"
QUIT_COMMANDS = {"quit", "exit", "q", "ออก"}


class CLI:
    """Presentation layer of the project (menus + input validation)."""

    def __init__(self, input_func=input, output_func=print):
        self.input_func = input_func
        self.output_func = output_func
        self.watchlist = []  # session-only in Sprint 1 (SQLite arrives in Sprint 3)
        self.running = False

    # ---------- display ----------

    def display_welcome_message(self):
        """Show the welcome banner."""
        self.output_func(WELCOME_MESSAGE)

    def display_menu(self):
        """Show the list of available commands."""
        self.output_func(MENU_TEXT)

    # ---------- input ----------

    def get_command_input(self):
        """Read one command line and normalise it (strip + lowercase)."""
        raw = self.input_func("movie> ")
        return raw.strip().lower()

    # ---------- main loop ----------

    def run(self):
        """Main loop: read -> validate -> dispatch, without ever crashing."""
        self.display_welcome_message()
        self.display_menu()
        self.running = True
        while self.running:
            try:
                self.handle_command(self.get_command_input())
            except EOFError:
                self.output_func("")
                self.output_func("[สิ้นสุดอินพุต]")
                self.running = False
            except KeyboardInterrupt:
                self.output_func("")
                self.output_func("[ยกเลิกโดยผู้ใช้]")
                self.running = False
            except ValueError as exc:
                self.output_func(f"คำเตือน: {exc}")
        self.output_func(FAREWELL_MESSAGE)

    def handle_command(self, text):
        """Validate and dispatch one normalised command line."""
        if not text:
            raise ValueError("กรุณาพิมพ์คำสั่ง (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)")
        parts = text.split()
        name, args = parts[0], parts[1:]
        if name in QUIT_COMMANDS:
            self.running = False
            return
        handlers = {
            "search": self.cmd_search,
            "discover": self.cmd_discover,
            "watchlist": self.cmd_watchlist,
            "help": self.cmd_help,
        }
        handler = handlers.get(name)
        if handler is None:
            raise ValueError(
                f"ไม่รู้จักคำสั่ง '{name}' (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)"
            )
        handler(args)

    # ---------- command handlers ----------

    def cmd_help(self, args):
        """help — show the command menu again."""
        self.display_menu()

    def cmd_search(self, args):
        """search <ชื่อเรื่อง> — find sample movies containing the text."""
        if not args:
            raise ValueError("คำสั่ง search ต้องระบุชื่อเรื่อง เช่น search alpha")
        query = " ".join(args)
        matches = [
            movie for movie in SAMPLE_MOVIES if query in movie["title"].lower()
        ]
        if not matches:
            self.output_func(
                f"ไม่พบภาพยนตร์ที่ตรงกับ '{query}' ในข้อมูลตัวอย่าง"
            )
            return
        self.output_func(f"พบ {len(matches)} เรื่อง:")
        for movie in matches:
            self.output_func("  " + self.format_movie(movie))

    def cmd_discover(self, args):
        """discover <รหัส> — list sample movies 'similar' to the seed id."""
        if not args:
            raise ValueError(
                "คำสั่ง discover ต้องระบุรหัสภาพยนตร์ เช่น discover 101"
            )
        seed_id = self.parse_positive_int(args[0], "รหัสภาพยนตร์")
        seed = next(
            (movie for movie in SAMPLE_MOVIES if movie["id"] == seed_id), None
        )
        if seed is None:
            self.output_func(f"ไม่พบรหัสภาพยนตร์ {seed_id} ในข้อมูลตัวอย่าง")
            return
        others = sorted(
            (m for m in SAMPLE_MOVIES if m["id"] != seed_id),
            key=lambda m: (m["vote_average"] or 0, m["vote_count"] or 0),
            reverse=True,
        )
        self.output_func(
            f"ภาพยนตร์ที่คล้ายกับ '{seed['title']}' (ข้อมูลตัวอย่าง):"
        )
        for rank, movie in enumerate(others[:5], start=1):
            self.output_func(f"  {rank}. {self.format_movie(movie)}")

    def cmd_watchlist(self, args):
        """watchlist add <รหัส> | list | clear — manage the watchlist."""
        if not args:
            raise ValueError(
                "คำสั่ง watchlist ต้องมีคำสั่งย่อย: add <รหัส> | list | clear"
            )
        action, rest = args[0], args[1:]
        if action == "list":
            self.show_watchlist()
        elif action == "add":
            self.add_to_watchlist(rest)
        elif action == "clear":
            self.watchlist.clear()
            self.output_func("ล้างรายการรับชมเรียบร้อยแล้ว")
        else:
            raise ValueError(
                f"ไม่รู้จักคำสั่งย่อย '{action}' (ใช้ add <รหัส> | list | clear)"
            )

    # ---------- helpers ----------

    def show_watchlist(self):
        """Print the current session watchlist."""
        if not self.watchlist:
            self.output_func("รายการรับชมยังว่างอยู่ (ใช้ watchlist add <รหัส>)")
            return
        self.output_func(f"รายการรับชม ({len(self.watchlist)} เรื่อง):")
        for movie in self.watchlist:
            self.output_func("  " + self.format_movie(movie))

    def add_to_watchlist(self, rest):
        """Add one sample movie id to the watchlist (no duplicates)."""
        if not rest:
            raise ValueError(
                "คำสั่ง watchlist add ต้องระบุรหัสภาพยนตร์ เช่น watchlist add 101"
            )
        movie_id = self.parse_positive_int(rest[0], "รหัสภาพยนตร์")
        movie = next((m for m in SAMPLE_MOVIES if m["id"] == movie_id), None)
        if movie is None:
            self.output_func(f"ไม่พบรหัสภาพยนตร์ {movie_id} ในข้อมูลตัวอย่าง")
            return
        if any(m["id"] == movie_id for m in self.watchlist):
            self.output_func(f"'{movie['title']}' อยู่ในรายการรับชมแล้ว")
            return
        self.watchlist.append(movie)
        self.output_func(f"เพิ่ม '{movie['title']}' เข้ารายการรับชมแล้ว")

    @staticmethod
    def parse_positive_int(text, label):
        """Convert text to a positive integer or raise a readable error."""
        try:
            value = int(text)
        except ValueError:
            raise ValueError(
                f"{label}ต้องเป็นตัวเลข (ได้รับ '{text}')"
            ) from None
        if value <= 0:
            raise ValueError(f"{label}ต้องมากกว่า 0 (ได้รับ {value})")
        return value

    @staticmethod
    def format_movie(movie):
        """Format one movie dict as a single display line."""
        year = (movie.get("release_date") or "")[:4] or "----"
        rating = movie.get("vote_average")
        rating_text = "-" if rating is None else f"{rating:.1f}"
        return f"[{movie['id']}] {movie['title']} ({year}) คะแนน {rating_text}"


Writing src/cli.py


In [9]:
%%writefile src/app.py
"""Entry point for the interactive Movie Discovery CLI.

Run with:

    python -m src.app
"""

from .cli import CLI


def main():
    """Start the interactive CLI."""
    CLI().run()


if __name__ == "__main__":
    main()


Writing src/app.py


In [10]:
%%writefile tests/conftest.py
"""Pytest configuration: make ``src`` and shared test helpers importable."""

import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
for path in (ROOT, ROOT / "tests"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


Writing tests/conftest.py


In [11]:
%%writefile tests/test_cli.py
"""Unit tests for the interactive CLI (Sprint 1 front-end).

Inputs are scripted (injected), so every edge case from the PLAN.md
Definition of Done is verified without a real terminal.
"""

import pytest

from src.cli import CLI


def scripted_inputs(lines):
    """Return an input function that replays the given lines, then EOF."""
    queue = list(lines)

    def fake_input(prompt=""):
        if not queue:
            raise EOFError
        return queue.pop(0)

    return fake_input


def run_cli(lines):
    """Run the CLI with scripted input; return (outputs, cli)."""
    outputs = []
    cli = CLI(input_func=scripted_inputs(lines), output_func=outputs.append)
    cli.run()
    return outputs, cli


def joined(outputs):
    return "\n".join(outputs)


@pytest.mark.parametrize("command", ["quit", "QUIT", "Quit", "  quit  ",
                                     "exit", "q", "ออก"])
def test_quit_variants_exit_with_farewell(command):
    outputs, cli = run_cli([command])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_eof_exits_gracefully():
    outputs, cli = run_cli([])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_empty_input_warns_and_continues():
    outputs, _ = run_cli(["", "quit"])
    assert "กรุณาพิมพ์คำสั่ง" in joined(outputs)
    assert "ลาก่อน" in joined(outputs)


def test_unknown_command_warns_and_recovers():
    outputs, _ = run_cli(["asdf", "quit"])
    text = joined(outputs)
    assert "ไม่รู้จักคำสั่ง" in text
    assert "help" in text
    assert "ลาก่อน" in text


def test_search_requires_a_title():
    outputs, _ = run_cli(["search", "quit"])
    assert "ต้องระบุชื่อเรื่อง" in joined(outputs)


def test_search_finds_sample_movie():
    outputs, _ = run_cli(["search alpha", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_search_is_case_insensitive():
    outputs, _ = run_cli(["SEARCH ALPHA", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_discover_rejects_non_numeric_id():
    outputs, _ = run_cli(["discover abc", "quit"])
    assert "ต้องเป็นตัวเลข" in joined(outputs)


def test_discover_rejects_zero_and_negative():
    outputs, _ = run_cli(["discover 0", "discover -3", "quit"])
    assert joined(outputs).count("ต้องมากกว่า 0") == 2


def test_discover_unknown_seed_is_reported():
    outputs, _ = run_cli(["discover 999", "quit"])
    assert "ไม่พบรหัสภาพยนตร์ 999" in joined(outputs)


def test_watchlist_add_list_and_duplicate():
    outputs, cli = run_cli(
        ["watchlist add 101", "watchlist list", "watchlist add 101", "quit"]
    )
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal'" in text
    assert "อยู่ในรายการรับชมแล้ว" in text
    assert len(cli.watchlist) == 1


def test_watchlist_needs_subcommand():
    outputs, _ = run_cli(["watchlist", "quit"])
    assert "ต้องมีคำสั่งย่อย" in joined(outputs)


def test_help_shows_menu():
    outputs, _ = run_cli(["help", "quit"])
    text = joined(outputs)
    assert "search" in text
    assert "quit" in text


def test_prompt_marker_is_stable():
    prompts = []

    def capture(prompt=""):
        prompts.append(prompt)
        raise EOFError

    cli = CLI(input_func=capture, output_func=lambda *_: None)
    cli.run()
    assert prompts and all(prompt == "movie> " for prompt in prompts)


Writing tests/test_cli.py


In [12]:
# สาธิตเซสชันการใช้งานจริง: จำลองอินพุตเหมือนผู้ใช้พิมพ์ทีละบรรทัด (ผลรันจริง)
from src.cli import CLI


def scripted(lines):
    """ฟังก์ชันอินพุตจำลอง: คืนค่าทีละบรรทัด แล้วส่ง EOF เมื่ออินพุตหมด"""
    queue = list(lines)

    def feed(prompt=""):
        if not queue:
            raise EOFError
        value = queue.pop(0)
        print(prompt + value)  # แสดงบรรทัดที่ "พิมพ์" เพื่อให้อ่าน transcript ได้
        return value

    return feed


demo_lines = ["search alpha", "discover 103",
              "watchlist add 101", "watchlist list", "quit"]
CLI(input_func=scripted(demo_lines), output_func=print).run()


  Movie Discovery & Watchlist Builder — CLI (Sprint 1)
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม


คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>      ค้นหาภาพยนตร์จากชื่อเรื่อง (ข้อมูลตัวอย่าง)
  discover <รหัส>         แสดงภาพยนตร์ที่คล้ายกัน (ข้อมูลตัวอย่าง)
  watchlist add <รหัส>    เพิ่มภาพยนตร์เข้ารายการรับชม
  watchlist list          แสดงรายการรับชม
  watchlist clear         ล้างรายการรับชม
  help                    แสดงคำสั่งทั้งหมด
  quit / exit / ออก       ออกจากโปรแกรม
movie> search alpha
พบ 1 เรื่อง:
  [101] Alpha Signal (2010) คะแนน 7.8
movie> discover 103
ภาพยนตร์ที่คล้ายกับ 'Cobalt Night' (ข้อมูลตัวอย่าง):
  1. [105] Emerald Run (2021) คะแนน 8.0
  2. [101] Alpha Signal (2010) คะแนน 7.8
  3. [102] Bravo Horizon (2014) คะแนน 6.1
  4. [104] Delta Echo (----) คะแนน 5.0
movie> watchlist add 101
เพิ่ม 'Alpha Signal' เข้ารายการรับชมแล้ว
movie> watchlist list
รายการรับชม (1 เรื่อง):
  [101] Alpha Signal (2010) คะแนน 7.8
movie> quit
ขอบคุณที่ใช้งานโปรแกรม ลาก่อน


In [13]:
# จำลองกรณีขอบเขตตาม Definition of Done (ผลรันจริงจากระบบ)
from src.cli import CLI


def scripted(lines):
    """ฟังก์ชันอินพุตจำลอง (เหมือนเซลล์ก่อนหน้า)"""
    queue = list(lines)

    def feed(prompt=""):
        if not queue:
            raise EOFError
        value = queue.pop(0)
        print(prompt + value)
        return value

    return feed


scenarios = [
    ("1) quit ตัวพิมพ์ใหญ่", ["QUIT"]),
    ("2) quit มีช่องว่างรอบ", ["   Quit   "]),
    ("3) บรรทัดว่าง", ["", "quit"]),
    ("4) คำสั่งไม่รู้จัก", ["asdf", "quit"]),
    ("5) รหัสไม่ใช่ตัวเลข", ["discover abc", "quit"]),
    ("6) รหัสติดลบ / ศูนย์", ["discover -3", "discover 0", "quit"]),
    ("7) อินพุตสิ้นสุด (EOF)", []),
]
for title, lines in scenarios:
    print("=" * 62)
    print(title)
    print("-" * 62)
    CLI(input_func=scripted(lines), output_func=print).run()


1) quit ตัวพิมพ์ใหญ่
--------------------------------------------------------------
  Movie Discovery & Watchlist Builder — CLI (Sprint 1)
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>      ค้นหาภาพยนตร์จากชื่อเรื่อง (ข้อมูลตัวอย่าง)
  discover <รหัส>         แสดงภาพยนตร์ที่คล้ายกัน (ข้อมูลตัวอย่าง)
  watchlist add <รหัส>    เพิ่มภาพยนตร์เข้ารายการรับชม
  watchlist list          แสดงรายการรับชม
  watchlist clear         ล้างรายการรับชม
  help                    แสดงคำสั่งทั้งหมด
  quit / exit / ออก       ออกจากโปรแกรม
movie> QUIT
ขอบคุณที่ใช้งานโปรแกรม ลาก่อน
2) quit มีช่องว่างรอบ
--------------------------------------------------------------
  Movie Discovery & Watchlist Builder — CLI (Sprint 1)
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>      ค้นหาภาพยนตร์จากชื่อเรื่อง (ข้อมูลตัวอย่าง)
  discover <รหัส>         แสดงภาพยนตร์ที่คล้ายกัน (ข้อมูลตัวอย่าง)
  watchlist add <รหัส>    เพิ่มภาพยนตร์เข้ารายการรับชม
  watchlist l

In [14]:
# รันชุดทดสอบอัตโนมัติของ CLI (จำลองอินพุต ไม่ต้องใช้ terminal จริง)
import sys
!{sys.executable} -m pytest -q


....................                                                     [100%]
20 passed in 0.05s


In [15]:
# ตรวจสอบมาตรฐานโค้ด (PEP 8)
import sys
!{sys.executable} -m flake8 src tests
print("ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)")


ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)


In [16]:
# ไฟล์ทั้งหมดที่สร้างขึ้นในรอบนี้
import pathlib

skip = {"__pycache__", ".pytest_cache", ".ipynb_checkpoints"}
for path in sorted(pathlib.Path(".").rglob("*")):
    if path.is_dir():
        continue
    if any(part in skip for part in path.parts):
        continue
    print("  " + path.relative_to(".").as_posix())


  PLAN.md
  requirements.txt
  setup.cfg
  src/__init__.py
  src/app.py
  src/cli.py
  src/sample_data.py
  tests/conftest.py
  tests/test_cli.py


## ส่วนที่ 3 — ผลลัพธ์และการทดสอบ (Result & Review)

### 3.1 สรุปความก้าวหน้าของงาน (Sprint Progress Summary)

- [x] ออกแบบโครงสร้างระบบและนิยาม Definition of Done ใน `PLAN.md`
- [x] พัฒนาชุดคำสั่งหลัก: `display_welcome_message` · `get_command_input` · เมนูและตัวจัดการคำสั่ง · ลูป `run`
- [x] ดักจับข้อผิดพลาดด้วย `try / except` (อินพุตผิดรูป · EOF · Ctrl+C)
- [x] ทดสอบอัตโนมัติ 20 เคสของ CLI (ผ่านทั้งหมด — เซลล์ pytest ด้านบน) + ทดสอบด้วยเซสชันสคริปต์
- [ ] ส่งมอบผ่าน Pull Request บน GitHub _(รอลิงก์ repository จากทีม)_

### 3.2 ผลการทดสอบกรณีขอบเขต (QA Report)

| รายการทดสอบ | อินพุต | ผลลัพธ์ที่คาดหวัง | ผลการทดสอบจริง | สถานะ |
|---|---|---|---|---|
| ออกจากโปรแกรม (ตัวพิมพ์ใหญ่) | `QUIT` | อำลาและหยุดทำงาน | แสดงข้อความอำลาและหลุดจากลูป | PASSED |
| ออกจากโปรแกรม (มีช่องว่างรอบ) | `   Quit   ` | อำลาและหยุดทำงาน | ตัดช่องว่างก่อนประมวลผล อำลาถูกต้อง | PASSED |
| คำสั่งไม่รู้จัก | `asdf` | แจ้งเตือน กลับเมนู | แจ้งเตือน "ไม่รู้จักคำสั่ง..." และกลับเมนู | PASSED |
| บรรทัดว่าง | (Enter เปล่า) | แจ้งเตือน ไม่ crash | แจ้งเตือน "กรุณาพิมพ์คำสั่ง..." | PASSED |
| รหัสไม่ใช่ตัวเลข | `discover abc` | แจ้งเตือน ไม่ crash | "รหัสภาพยนตร์ต้องเป็นตัวเลข..." | PASSED |
| รหัสติดลบ / ศูนย์ | `discover -3`, `0` | ปฏิเสธพร้อมข้อความ | "รหัสภาพยนตร์ต้องมากกว่า 0..." | PASSED |
| ค้นหาปกติ | `search alpha` | พบภาพยนตร์ | พบ 1 เรื่อง (Alpha Signal) | PASSED |
| ป้องกันเพิ่มซ้ำ | `watchlist add 101` ×2 | เพิ่มครั้งเดียว | ครั้งที่สองแจ้ง "อยู่ในรายการรับชมแล้ว" | PASSED |
| อินพุตสิ้นสุด | EOF | ออกอย่างสุภาพ | แสดง "[สิ้นสุดอินพุต]" + ข้อความอำลา | PASSED |

### 3.3 สรุปบทเรียน (Retrospective: Wow! & Whoops!)

**Wow! (ส่วนที่ทำได้ดี):** แยกส่วนนำเสนอ (CLI) ออกจากส่วนอื่นอย่างชัดเจน ทำให้ทดสอบง่าย ·
การตรวจสอบอินพุตครอบคลุมทุกกรณีใน Definition of Done · มีชุดทดสอบอัตโนมัติ 20 เคส
ที่จำลองอินพุตได้จริง ทำให้ทุก DoD พิสูจน์ซ้ำได้ด้วยคำสั่งเดียว

**Whoops! (ปัญหาและแนวทางแก้ไข):** เดิม CLI ถูกออกแบบเป็นคำสั่งแบบครั้งเดียว (argparse)
แต่สเปก Sprint 1 ต้องการเมนูแบบโต้ตอบพร้อมตรวจสอบอินพุต — จึงรีแฟกเตอร์มาเป็นคลาส `CLI`
ที่แยกส่วนนำเสนอชัดเจน · รายการรับชมยังเป็นข้อมูลชั่วคราวในหน่วยความจำ (ปิดโปรแกรมแล้วหาย)
— กำหนดแนวทางแก้เป็นฐานข้อมูล SQLite ใน Sprint 3

### 3.4 ลิงก์และขั้นตอนถัดไป

- **Repository / Pull Request:** _(จะเพิ่มเมื่อสร้างบน GitHub)_
- **Sprint 2 (Back-End):** เชื่อมต่อ TMDB API จริง · SQLite · ค้นหา/กรอง/เรียงลำดับ · File I/O


---

*ข้อมูลภาพยนตร์ใน Sprint 1 เป็นข้อมูลตัวอย่างสำหรับสาธิตส่วนติดต่อผู้ใช้ — การเชื่อมต่อ TMDB API จริงเริ่มใน Sprint 2
(เมื่อเชื่อมแล้วจะแสดงข้อความ "This product uses the TMDB API but is not endorsed or certified by TMDB." ในโปรแกรมและเอกสาร)*

*จัดทำเพื่อส่งงาน Sprint 1 รายวิชา CP352301 Script Programming (1/2569) — เซลล์ทั้งหมดรันจริงครบวงจร กด "Run all" เพื่อทำซ้ำได้ทันที*
